# 🚀 Zalo AI Challenge 2026 — LifeMate: Pro Pipeline Runner (Fine-tune + Inference)

Kiến trúc **Decoupled Thin-Runner** chuẩn thi đấu AI chuyên nghiệp:
- **Bỏ qua ASR:** Nạp sẵn cache 1,933 file audio trong 0.1 giây (0 GB VRAM).
- **QLoRA SFT:** Fine-tune `Qwen2.5-3B-Instruct` trên 1,545 dialogues (`train.jsonl`), lưu adapter thẳng vào Google Drive.
- **SLM 4-bit NF4:** Chiếm chỉ **2.1 GB VRAM** (trống >12.5 GB, 0% nguy cơ OOM).
- **Auto-resume & Guardrails:** Tự động sửa ID/source theo DB state, sao lưu lũy tiến ra Google Drive.

In [ ]:
# 1. Mount Google Drive & Cài đặt thư viện
from google.colab import drive
drive.mount('/content/drive')

import torch
assert torch.cuda.is_available(), 'VUI LÒNG BẬT GPU: Vào Runtime -> Change runtime type -> Chọn T4 GPU!'
print(f'GPU đang dùng: {torch.cuda.get_device_name(0)}')

!pip install -q transformers accelerate bitsandbytes scipy rapidfuzz peft datasets


In [ ]:
# 2. Kéo mã nguồn mới nhất từ GitHub
import os
if not os.path.exists('/content/ZaloAIC-Lifemate'):
    !git clone https://github.com/duylhm/ZaloAIC-Lifemate.git /content/ZaloAIC-Lifemate
else:
    !cd /content/ZaloAIC-Lifemate && git pull origin main

%cd /content/ZaloAIC-Lifemate


In [ ]:
# 3. HUẤN LUYỆN FINE-TUNE QLoRA (Chạy 1 lần, ~10-15 phút trên T4 GPU)
# Script sẽ học 1,545 mẫu từ train.jsonl và lưu adapter thẳng vào:
# /content/drive/MyDrive/qwen2.5_3b_lifemate_lora
!python train_sft.py


In [ ]:
# 4. CHẠY SUY LUẬN VỚI LORA ADAPTER ĐÃ FINE-TUNE (~20-30 phút)
# Tự động nạp LoRA adapter từ Drive và sinh submission.jsonl độ chính xác cao
# Cờ --force đảm bảo xóa bỏ file 6 điểm cũ và chạy mới 100%
!python run_inference.py --force


In [ ]:
# 5. Tải file kết quả submission.jsonl về máy tính
from google.colab import files
import os

if os.path.exists('/content/submission.jsonl'):
    print('Đang tải file submission.jsonl về máy tính...')
    files.download('/content/submission.jsonl')
else:
    print('File submission đã được lưu an toàn tại: /content/drive/MyDrive/submission.jsonl')
